# Notebook 3: Price/Time-Series Modality
### Simple comparison between Rogendo/forex-lstm-models vs custom LSTM vs custom GRU
### Asset: EURUSD 1h | Metric focus: Directional Accuracy
1. Rogendo/forex-lstm-models: pretrained dual-output LSTM models trained on forex OHLCV data; predicts next-step price movement and directional probability for specific currency pair and timeframe combinations
2. custom LSTM: self-trained LSTM model following a similar sequence forecasting pipeline using historical EURUSD 1h data for directional prediction
3. custom GRU: self-trained GRU-based sequence model for EURUSD 1h directional forecasting; uses a more lightweight recurrent architecture compared to LSTM with potentially faster training and competitive performance

### Step 0 - Basic check that we are in virtual environment

In [1]:
import sys
print(sys.executable)

C:\Users\KJ\Documents\School\FYP\Feature Prototype ENV\.venv\Scripts\python.exe


### Step 1 - Install dependencies and Imports

In [2]:
# Install dependencies
# * my dependencies are already installed directly in venv through terminal so it is commented out
# * tensorflow requires 3.13 or below python version, so either a downgrade from latest 3.14 or separate version needs to be installed

# %pip install yfinance tensorflow scikit-learn huggingface_hub --quiet

In [3]:
# Imports
import logging
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score
from huggingface_hub import hf_hub_download
import yfinance as yf
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from absl import logging as absl_logging
import warnings
warnings.filterwarnings('ignore')
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
absl_logging.set_verbosity(absl_logging.ERROR)

### Step 2 - Fetch real data from yfinance
- 2 years worth of EURUSD 1hr data into a dataframe

In [4]:
# Fetch EURUSD 1h data from yfinance (2 years)
print("Fetching EURUSD 1h data from yfinance...")
df_raw = yf.download("EURUSD=X", period="2y", interval="1h", auto_adjust=True)
df_raw.dropna(inplace=True)

print(f"Data shape: {df_raw.shape}")
print(f"Date range: {df_raw.index[0]} to {df_raw.index[-1]}")
print(df_raw.tail())

Fetching EURUSD 1h data from yfinance...


[*********************100%***********************]  1 of 1 completed

Data shape: (12319, 5)
Date range: 2024-05-22 23:00:00+00:00 to 2026-05-22 21:00:00+00:00
Price                         Close      High       Low      Open   Volume
Ticker                     EURUSD=X  EURUSD=X  EURUSD=X  EURUSD=X EURUSD=X
Datetime                                                                  
2026-05-22 17:00:00+00:00  1.161710  1.162250  1.161305  1.161575        0
2026-05-22 18:00:00+00:00  1.161575  1.161845  1.161305  1.161710        0
2026-05-22 19:00:00+00:00  1.161036  1.161710  1.160901  1.161575        0
2026-05-22 20:00:00+00:00  1.160497  1.161036  1.160362  1.161036        0
2026-05-22 21:00:00+00:00  1.160497  1.160631  1.160497  1.160631        0


### Step 3 - Prepocess and building of sequence
- Earlier results showed that yfinance volume data for forex is not meaningful / mostly empty
- However, Volume is still retained to maintain compatibility with the pretrained Rogendo LSTM model input shape (for this notebook comparison at least)
- Lookback is set to 15 to mirror Rogendo's preprocessing: RobustScaler, 15 lookback steps, 5 features (OHLCV)
- https://huggingface.co/Rogendo/forex-lstm-models

In [5]:
LOOKBACK = 15
FEATURES = ['Open', 'High', 'Low', 'Close', 'Volume']

data = df_raw[FEATURES].values

scaler = RobustScaler()
data_scaled = scaler.fit_transform(data)

def build_sequences(data, lookback):
    X, y_price, y_dir = [], [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i])
        # Price change: next close - current close (scaled)
        price_change = data[i, 3] - data[i - 1, 3]
        y_price.append(price_change)
        # Direction: 1 if price goes up, 0 if down
        y_dir.append(1 if price_change > 0 else 0)
    return np.array(X), np.array(y_price), np.array(y_dir)

X, y_price, y_dir = build_sequences(data_scaled, LOOKBACK)

# 80/20 train/val split (no shuffle as time series order matters)
split = int(len(X) * 0.8)
X_train, X_val = X[:split], X[split:]
yp_train, yp_val = y_price[:split], y_price[split:]
yd_train, yd_val = y_dir[:split], y_dir[split:]

print(f"Train samples: {len(X_train)} | Val samples: {len(X_val)}")
print(f"Input shape: {X_train.shape}")

Train samples: 9843 | Val samples: 2461
Input shape: (9843, 15, 5)


### Step 4 - Build custom LSTM model
- alternative to pretrained model
- mirrors closely to Rogendo architecture

In [6]:
def build_lstm(input_shape):
    inputs = Input(shape=input_shape)
    x = LSTM(50, return_sequences=True)(inputs)
    x = Dropout(0.2)(x)
    x = LSTM(25, return_sequences=False)(x)
    x = Dropout(0.2)(x)
    x = Dense(20, activation='relu')(x)
    out_price = Dense(1, name='price')(x)
    out_dir = Dense(1, activation='sigmoid', name='direction')(x)
    model = Model(inputs, [out_price, out_dir])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss={'price': 'mse', 'direction': 'binary_crossentropy'},
        metrics={'direction': 'accuracy'}
    )
    return model

lstm_model = build_lstm((LOOKBACK, len(FEATURES)))
lstm_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 15, 5)             │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm (LSTM)                   │ (None, 15, 50)            │          11,200 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 15, 50)            │               0 │ lstm[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_1 (LSTM)                 │ (None, 25)                │           7,600 │ dropout[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 25)                │               0 │ lstm_1[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 20)                │             520 │ dropout_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ price (Dense)                 │ (None, 1)                 │              21 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ direction (Dense)             │ (None, 1)                 │              21 │ dense[0][0]                │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 19,362 (75.63 KB)

 Trainable params: 19,362 (75.63 KB)

 Non-trainable params: 0 (0.00 B)

### Step 5 - Build custom GRU model
- alternative to pretrained model and ealier lstm model

In [7]:
def build_gru(input_shape):
    inputs = Input(shape=input_shape)
    x = GRU(50, return_sequences=True)(inputs)
    x = Dropout(0.2)(x)
    x = GRU(25, return_sequences=False)(x)
    x = Dropout(0.2)(x)
    x = Dense(20, activation='relu')(x)
    out_price = Dense(1, name='price')(x)
    out_dir = Dense(1, activation='sigmoid', name='direction')(x)
    model = Model(inputs, [out_price, out_dir])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss={'price': 'mse', 'direction': 'binary_crossentropy'},
        metrics={'direction': 'accuracy'}
    )
    return model

gru_model = build_gru((LOOKBACK, len(FEATURES)))
gru_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 15, 5)             │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gru (GRU)                     │ (None, 15, 50)            │           8,550 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_2 (Dropout)           │ (None, 15, 50)            │               0 │ gru[0][0]                  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gru_1 (GRU)                   │ (None, 25)                │           5,775 │ dropout_2[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_3 (Dropout)           │ (None, 25)                │               0 │ gru_1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 20)                │             520 │ dropout_3[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ price (Dense)                 │ (None, 1)                 │              21 │ dense_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ direction (Dense)             │ (None, 1)                 │              21 │ dense_1[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 14,887 (58.15 KB)

 Trainable params: 14,887 (58.15 KB)

 Non-trainable params: 0 (0.00 B)

### Step 6 - Train both custom models

In [8]:
es = EarlyStopping(patience=15, restore_best_weights=True, verbose=1)

print("Training custom LSTM...")
lstm_history = lstm_model.fit(
    X_train, {'price': yp_train, 'direction': yd_train},
    validation_data=(X_val, {'price': yp_val, 'direction': yd_val}),
    epochs=50,
    batch_size=32,
    callbacks=[es],
    verbose=1
)

print("\nTraining custom GRU...")
gru_history = gru_model.fit(
    X_train, {'price': yp_train, 'direction': yd_train},
    validation_data=(X_val, {'price': yp_val, 'direction': yd_val}),
    epochs=50,
    batch_size=32,
    callbacks=[es],
    verbose=1
)

Training custom LSTM...
Epoch 1/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - direction_accuracy: 0.5347 - direction_loss: 0.6914 - loss: 0.6921 - price_loss: 6.8239e-04 - val_direction_accuracy: 0.5595 - val_direction_loss: 0.6873 - val_loss: 0.6878 - val_price_loss: 5.4326e-04
Epoch 2/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - direction_accuracy: 0.5380 - direction_loss: 0.6905 - loss: 0.6909 - price_loss: 3.6175e-04 - val_direction_accuracy: 0.5595 - val_direction_loss: 0.6869 - val_loss: 0.6871 - val_price_loss: 1.4003e-04
Epoch 3/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - direction_accuracy: 0.5388 - direction_loss: 0.6904 - loss: 0.6907 - price_loss: 2.6608e-04 - val_direction_accuracy: 0.5595 - val_direction_loss: 0.6869 - val_loss: 0.6871 - val_price_loss: 1.9878e-04
Epoch 4/50
308/308 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - direction_accuracy: 0.5388 - direction_loss: 0.6903 - loss: 0.6905 - price_loss: 2.3711e-04 - val_direction_accuracy: 0.5595 - val_direction_loss: 0.6

### Step 7 - Load pretrained model (Rogendo/forex-lstm-models)
- usage and implementation can be seen https://huggingface.co/Rogendo/forex-lstm-models

In [9]:
print("Downloading Rogendo EURUSD 1h pretrained model...")
pretrained_path = hf_hub_download(repo_id="Rogendo/forex-lstm-models", filename="EURUSD_X_1h_model.h5")
scaler_path = hf_hub_download(repo_id="Rogendo/forex-lstm-models", filename="EURUSD_X_1h_features.pkl")

pretrained_model = load_model(pretrained_path)

with open(scaler_path, 'rb') as f:
    feature_info = pickle.load(f)

# NOTE: model card states lookback=15 but actual saved model expects 25
# * using feature_info['lookback'] at runtime to be safe rather than hardcoding
PRETRAINED_LOOKBACK = feature_info['lookback']
print(f"Pretrained model loaded | Lookback: {feature_info['lookback']}")
pretrained_model.summary()

Pretrained model loaded | Lookback: 25


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 25, 5)             │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_2 (LSTM)                 │ (None, 25, 50)            │          11,200 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_3 (Dropout)           │ (None, 25, 50)            │               0 │ lstm_2[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_3 (LSTM)                 │ (None, 25)                │           7,600 │ dropout_3[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_4 (Dropout)           │ (None, 25)                │               0 │ lstm_3[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 20)                │             520 │ dropout_4[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_5 (Dropout)           │ (None, 20)                │               0 │ dense_2[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_3 (Dense)               │ (None, 10)                │             210 │ dropout_5[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ price_prediction (Dense)      │ (None, 1)                 │              11 │ dense_3[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ direction_prediction (Dense)  │ (None, 1)                 │              11 │ dense_3[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 19,554 (76.39 KB)

 Trainable params: 19,552 (76.38 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

### Step 8 - Evaluate all 3 models on validation set
- Custom LSTM and GRU use LOOKBACK=15 (our own choice, trained above)
- pretrained model uses PRETRAINED_LOOKBACK from pickle (actual model expects 25, despite model card stating 15 - documented inconsistency, see Cell 12 notes)

In [10]:
X_pre, _, yd_pre = build_sequences(data_scaled, PRETRAINED_LOOKBACK)
split_pre = int(len(X_pre) * 0.8)
X_val_pre = X_pre[split_pre:]
yd_val_pre = yd_pre[split_pre:]

def evaluate_model(model, X, yd, label):
    preds = model.predict(X, verbose=0)
    dir_probs = preds[1].flatten()
    dir_preds = (dir_probs > 0.5).astype(int)
    acc = accuracy_score(yd, dir_preds)
    avg_prob = float(np.mean(dir_probs))
    signal = 1 if avg_prob > 0.5 else -1
    print(f" {label:<30} Directional Accuracy: {acc*100:.2f}%  Signal: {signal:+d}  Conf: {avg_prob:.3f}")
    return acc, signal, avg_prob

print("Evaluation on Validation Set:")
lstm_acc, lstm_signal, lstm_conf = evaluate_model(lstm_model, X_val, yd_val, "Custom LSTM")
gru_acc, gru_signal, gru_conf = evaluate_model(gru_model, X_val, yd_val, "Custom GRU")
pretrained_acc, pretrained_signal, pretrained_conf = evaluate_model(pretrained_model, X_val_pre, yd_val_pre, "Rogendo Pretrained LSTM")

Evaluation on Validation Set:
 Custom LSTM                    Directional Accuracy: 55.95%  Signal: -1  Conf: 0.438
 Custom GRU                     Directional Accuracy: 55.95%  Signal: -1  Conf: 0.479
 Rogendo Pretrained LSTM        Directional Accuracy: 55.96%  Signal: -1  Conf: 0.459


### Step 9 - Final observation + decision

Observation:
- All three sequence models produced very similar directional validation accuracy (~56%)
- All models generated the same final directional signal (-1) on the validation evaluation
- Confidence outputs between the GRU and pretrained LSTM were also closely aligned
- The pretrained model required a different lookback window than documented in its model card, which introduced additional implementation inconsistency during integration

Chosen model for pipeline and reasoning: **Custom GRU**
- Both custom recurrent models achieved comparable performance to the pretrained model on the validation dataset
- The GRU architecture provides similar sequence modelling capability while using a simpler gating structure compared to LSTM
- The custom GRU model also integrates more cleanly into the prototype pipeline since the training configuration and preprocessing steps are fully controlled within the project environment
- Due to the nearly identical validation performance across all three models, the GRU was selected as the primary sequence model for the prototype implementation